# 병합 control48 추론만 실행

Teacher/E5/원본 CSV를 읽거나 학습하지 않고 병합 Student+원본 Head 패키지만 로딩합니다.
먼저 01 노트북에서 저장한 control48.pt를 지정하세요. 원래 Head의 5유형 argmax만 최종 3유형으로 표시합니다.


In [ ]:
from pathlib import Path
import os, sys
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
WORKSPACE_PATH = None
SOURCE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'src/heads/distilation').is_dir()), None)
if SOURCE_ROOT is None:
    raise FileNotFoundError('GEVIS 프로젝트 또는 notebooks/distilation에서 실행하세요.')
ROOT = SOURCE_ROOT if WORKSPACE_PATH is None else Path(WORKSPACE_PATH).expanduser().resolve()
SOURCE = SOURCE_ROOT / 'src'
for name, module in tuple(sys.modules.items()):
    if name in ('heads', 'pecvd') or name.startswith(('heads.', 'pecvd.')):
        for location in [getattr(module, '__file__', None), *getattr(module, '__path__', ())]:
            if location and not Path(location).resolve().is_relative_to(SOURCE):
                raise RuntimeError('다른 프로젝트 모듈이 로딩되어 있습니다. 해당 커널의 결과를 보존한 뒤 새 커널을 사용하세요.')
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from heads import distilation as kd


In [ ]:
MERGED_CHECKPOINT = None  # 01의 MERGED_CHECKPOINT 경로
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if MERGED_CHECKPOINT is None:
    CANDIDATES = sorted((ROOT / 'artifacts/gevis_distilation').glob('*/control48.pt'))
    display(pd.DataFrame({'checkpoint': [str(p.relative_to(ROOT)) for p in CANDIDATES]}))
    if not CANDIDATES:
        raise FileNotFoundError('01에서 병합본을 먼저 저장하거나 MERGED_CHECKPOINT를 지정하세요.')
    if len(CANDIDATES) == 1:
        MERGED_CHECKPOINT = CANDIDATES[0]
    else:
        INDEX = int(input('사용할 병합본 번호: '))
        if not 0 <= INDEX < len(CANDIDATES):
            raise ValueError('표의 번호를 입력하세요.')
        MERGED_CHECKPOINT = CANDIDATES[INDEX]
else:
    MERGED_CHECKPOINT = Path(MERGED_CHECKPOINT)
    if not MERGED_CHECKPOINT.is_absolute():
        MERGED_CHECKPOINT = ROOT / MERGED_CHECKPOINT
MERGED = kd.load_merged(MERGED_CHECKPOINT, device=DEVICE)
# Reuse the Validation-selected execution batch if this kernel ran notebook 01.
# Otherwise 32 is a starting setting; verify it on Validation with notebook 01.
import heads.control48_fast as fast_runtime
BATCH_SIZE = globals().get('EXECUTION_SELECTION', {}).get('center_batch_size', 32)
SESSION = fast_runtime.FastSession(MERGED, center_batch_size=BATCH_SIZE)
display(kd.mapping_table())


## 입력 연결

이 셀은 기본적으로 실행하지 않습니다. REQUEST에 실제 raw Window와 native 문맥을 넣은 뒤 RUN_REQUEST=True로 바꾸세요.
예: current=[time,sensors], step, recipe_key, loop_key, offset_samples, lineage_id, sensor_columns.
standardized=False이면 패키지의 TRAIN 정규화를 사용합니다. 새로운 데이터에 mean/std·MIN/MAX를 다시 맞추지 않습니다.
raw OPEN/CLOSE 밸브 인코딩은 원본 domain 패키지의 입력 규약을 그대로 따릅니다.
아래는 파일 읽기·모델 fitting·무작위 입력 생성 없이 명시한 요청만 처리합니다.


In [ ]:
RUN_REQUEST = False
REQUEST = None
if RUN_REQUEST:
    if not isinstance(REQUEST, dict):
        raise ValueError('실제 native Window 요청을 REQUEST에 지정하세요.')
    REPLY = SESSION.process_window(**REQUEST)
    print(REPLY['status'], REPLY.get('reason'))
    display(pd.DataFrame(REPLY['answers']))
